# YOLO vs RT-DETR — Results Analysis

Notebook for generating thesis figures from experiment results.

Sections:
1. mAP comparison across models and datasets
2. Speed vs accuracy trade-off (FPS vs mAP)
3. Per-class AP analysis
4. Qualitative comparison (side-by-side detections)
5. RT-DETR attention visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'figure.dpi': 120,
})

FIG_DIR = Path('thesis_figures')
FIG_DIR.mkdir(exist_ok=True)

## 1. Load experiment results

In [ ]:
# Load CSV produced by run_experiments.py
# If experiments are not done yet, use the mock data below

try:
    df = pd.read_csv('../runs/experiment_results.csv')
    print('Loaded real results')
except FileNotFoundError:
    print('Using mock data for notebook development')
    df = pd.DataFrame([
        {'experiment': 'yolov8l_bdd100k',  'eval_dataset': 'bdd100k', 'mAP50': 0.612, 'mAP50-95': 0.401, 'fps': 94},
        {'experiment': 'yolov8x_bdd100k',  'eval_dataset': 'bdd100k', 'mAP50': 0.635, 'mAP50-95': 0.423, 'fps': 62},
        {'experiment': 'yolov10l_bdd100k',  'eval_dataset': 'bdd100k', 'mAP50': 0.628, 'mAP50-95': 0.415, 'fps': 88},
        {'experiment': 'rtdetr_l_bdd100k', 'eval_dataset': 'bdd100k', 'mAP50': 0.641, 'mAP50-95': 0.432, 'fps': 74},
        {'experiment': 'rtdetr_x_bdd100k', 'eval_dataset': 'bdd100k', 'mAP50': 0.658, 'mAP50-95': 0.449, 'fps': 41},
        # Cross-dataset: trained BDD100K, tested on KITTI
        {'experiment': 'yolov8l_bdd100k',  'eval_dataset': 'kitti',   'mAP50': 0.571, 'mAP50-95': 0.362, 'fps': 94},
        {'experiment': 'rtdetr_l_bdd100k', 'eval_dataset': 'kitti',   'mAP50': 0.598, 'mAP50-95': 0.389, 'fps': 74},
    ])

df.head(10)

## 2. mAP Comparison

In [ ]:
from utils.visualization import plot_model_comparison

for dataset in df['eval_dataset'].unique():
    sub = df[df['eval_dataset'] == dataset].set_index('experiment')
    results = sub[['mAP50', 'mAP50-95']].to_dict(orient='index')

    fig = plot_model_comparison(
        results, metric='mAP50-95',
        title=f'mAP@[.5:.95] — Evaluated on {dataset.upper()}'
    )
    path = FIG_DIR / f'map_comparison_{dataset}.pdf'
    fig.savefig(path, bbox_inches='tight')
    print(f'Saved {path}')
    plt.show()

## 3. Speed vs Accuracy

In [ ]:
MODEL_COLORS = {
    'yolov8l_bdd100k':  '#0072B2',
    'yolov8x_bdd100k':  '#56B4E9',
    'yolov10l_bdd100k': '#009E73',
    'rtdetr_l_bdd100k': '#D55E00',
    'rtdetr_x_bdd100k': '#CC79A7',
}

MODEL_LABELS = {
    'yolov8l_bdd100k':  'YOLOv8-L',
    'yolov8x_bdd100k':  'YOLOv8-X',
    'yolov10l_bdd100k': 'YOLOv10-L (NMS-free)',
    'rtdetr_l_bdd100k': 'RT-DETR-L',
    'rtdetr_x_bdd100k': 'RT-DETR-X',
}

sub = df[df['eval_dataset'] == 'bdd100k'].copy()

fig, ax = plt.subplots(figsize=(9, 6))
for _, row in sub.iterrows():
    name = row['experiment']
    ax.scatter(
        row.get('fps', np.nan), row['mAP50-95'],
        s=160, color=MODEL_COLORS.get(name, 'gray'),
        edgecolors='black', linewidths=0.8, zorder=3
    )
    ax.annotate(
        MODEL_LABELS.get(name, name),
        (row.get('fps', np.nan), row['mAP50-95']),
        textcoords='offset points', xytext=(6, 4), fontsize=9
    )

ax.set_xlabel('FPS (V100 GPU, batch=1)')
ax.set_ylabel('mAP@[.5:.95]')
ax.set_title('Speed–Accuracy Trade-off on BDD100K')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'speed_accuracy.pdf', bbox_inches='tight')
plt.show()

## 4. RT-DETR Decoder Attention Visualization

In [ ]:
# Requires a trained RT-DETR checkpoint
# and the HuggingFace backend (backend='huggingface')

import cv2
from omegaconf import OmegaConf
from models.rtdetr.rtdetr_model import RTDETRDetector
from utils.visualization import visualize_decoder_attention, draw_detections

CHECKPOINT = '../runs/rtdetr_l_bdd100k/weights/best.pt'   # update path
IMAGE_PATH  = '../data/kitti/training/image_2/000001.png' # update path
CLASSES     = ['car', 'truck', 'bus', 'motorcycle', 'bicycle', 'pedestrian']

# Load model
model = RTDETRDetector(
    variant='rtdetr-l',
    num_classes=len(CLASSES),
    backend='huggingface',      # needed for attention extraction
    checkpoint=CHECKPOINT,
)

img_bgr = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

preds = model.predict([img_rgb], conf=0.3, device='cpu')[0]
print(f"Detected {len(preds['boxes'])} objects")

In [ ]:
# Visualize attention for top-3 detections
n_show = min(3, len(preds['boxes']))
fig, axes = plt.subplots(1, n_show + 1, figsize=(5 * (n_show + 1), 5))

# Original with boxes
h, w = img_rgb.shape[:2]
boxes_abs = preds['boxes'].clone()
boxes_abs[:, [0,2]] *= w
boxes_abs[:, [1,3]] *= h
annotated = draw_detections(img_rgb, boxes_abs, preds['labels'],
                            preds['scores'], class_names=CLASSES, is_bgr=False)
axes[0].imshow(annotated)
axes[0].set_title('Detections')
axes[0].axis('off')

# Attention overlays
for i in range(n_show):
    overlay = visualize_decoder_attention(
        img_rgb,
        preds['decoder_attentions'],   # extracted from HF model
        query_idx=i,
        alpha=0.55,
    )
    cls_name = CLASSES[int(preds['labels'][i])]
    score = float(preds['scores'][i])
    axes[i+1].imshow(overlay)
    axes[i+1].set_title(f'Query {i}: {cls_name} ({score:.2f})')
    axes[i+1].axis('off')

plt.suptitle('RT-DETR Decoder Cross-Attention', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'rtdetr_attention.pdf', bbox_inches='tight')
plt.show()